# Violence Detection Model Training - Pose Estimation Based

This notebook trains a **pose-based violence detection model** using skeleton keypoints extracted from video data in S3.

## Architecture:
- **Person Detection**: YOLOv8 to detect people in each frame
- **Pose Estimation**: Extract 17 keypoints (COCO format) for each person
- **Temporal Modeling**: Spatial-Temporal GCN + LSTM for action recognition
- **Classification**: FC layer for violence detection

## Advantages over RGB/Flow:
- More efficient (skeleton sequences are compact)
- Better occlusion handling
- Privacy-preserving (no visual data)
- Lower computational cost for real-time deployment

## Requirements:
- Videos stored in S3 bucket
- Videos organized as: `s3://bucket-name/violence/` and `s3://bucket-name/normal/`
- Each video ~10 seconds duration

## 1. Setup and Imports

In [ ]:
# Install required packages
!pip install -q boto3 sagemaker opencv-python-headless torch torchvision
!pip install -q pandas numpy scikit-learn matplotlib seaborn tqdm
!pip install -q ultralytics  # For YOLOv8
!pip install -q torch-geometric  # For GCN operations
!pip install -q tensorboard

In [ ]:
import os
import sys
import json
import random
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import boto3
import sagemaker
from sagemaker import get_execution_role

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

from ultralytics import YOLO

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Configuration

In [ ]:
# AWS Configuration
sagemaker_session = sagemaker.Session()
role = get_execution_role()
bucket = 'nest-vision'  # or specify your bucket name
prefix = 'дата'  # folder in S3 bucket

print(f"SageMaker Role: {role}")
print(f"Default S3 Bucket: {bucket}")
print(f"S3 Prefix: {prefix}")

# Initialize S3 client
s3_client = boto3.client('s3')
s3_resource = boto3.resource('s3')

In [ ]:
# Model Configuration
CONFIG = {
    # S3 paths
    'S3_BUCKET': bucket,
    'S3_VIOLENCE_PREFIX': 'дата/set1/Aggression-from-adult/',
    'S3_NORMAL_PREFIX': 'дата/Subset0/normal/',
    'S3_MODEL_OUTPUT': f's3://{bucket}/{prefix}/models/',
    
    # Local paths
    'LOCAL_DATA_DIR': '/home/ec2-user/SageMaker/violence_detection_data',
    'LOCAL_VIOLENCE_DIR': '/home/ec2-user/SageMaker/violence_detection_data/violence',
    'LOCAL_NORMAL_DIR': '/home/ec2-user/SageMaker/violence_detection_data/normal',
    'MODEL_SAVE_DIR': '/home/ec2-user/SageMaker/pose-models',
    'LOGS_DIR': '/home/ec2-user/SageMaker/pose-logs',
    'CACHE_DIR': '/home/ec2-user/SageMaker/pose-cache',  # Cache for extracted poses
    
    # Video processing
    'SEQUENCE_LENGTH': 30,  # Number of frames per sequence (1 second at 30fps)
    'FPS_SAMPLE': 30,  # Sample all frames for pose extraction
    'MAX_PERSONS': 2,  # Maximum number of people to track per frame
    
    # Pose configuration
    'NUM_KEYPOINTS': 17,  # COCO format keypoints
    'KEYPOINT_DIM': 3,  # (x, y, confidence)
    'POSE_CONFIDENCE_THRESHOLD': 0.3,  # Minimum confidence for valid keypoint
    
    # Training
    'BATCH_SIZE': 16,  # Can be larger for pose data (smaller than images)
    'NUM_EPOCHS': 50,
    'LEARNING_RATE': 1e-3,
    'WEIGHT_DECAY': 1e-4,
    'TRAIN_VAL_SPLIT': 0.85,
    'RANDOM_SEED': 42,
    
    # Model architecture
    'GCN_HIDDEN_DIM': 64,  # Hidden dimension for GCN
    'LSTM_HIDDEN_DIM': 128,
    'LSTM_LAYERS': 2,
    'NUM_CLASSES': 2,
    'DROPOUT': 0.3,
    
    # Device
    'DEVICE': 'cuda' if torch.cuda.is_available() else 'cpu',
    'NUM_WORKERS': 8,  # More workers for lightweight pose data
}

# Create directories
for dir_path in [CONFIG['LOCAL_DATA_DIR'], CONFIG['LOCAL_VIOLENCE_DIR'], 
                 CONFIG['LOCAL_NORMAL_DIR'], CONFIG['MODEL_SAVE_DIR'], 
                 CONFIG['LOGS_DIR'], CONFIG['CACHE_DIR']]:
    os.makedirs(dir_path, exist_ok=True)

# Set random seeds
random.seed(CONFIG['RANDOM_SEED'])
np.random.seed(CONFIG['RANDOM_SEED'])
torch.manual_seed(CONFIG['RANDOM_SEED'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG['RANDOM_SEED'])

print(json.dumps(CONFIG, indent=2))

## 3. COCO Skeleton Definition

Define the skeleton connectivity for graph construction

In [ ]:
# COCO keypoints (17 keypoints)
COCO_KEYPOINTS = [
    'nose',           # 0
    'left_eye',       # 1
    'right_eye',      # 2
    'left_ear',       # 3
    'right_ear',      # 4
    'left_shoulder',  # 5
    'right_shoulder', # 6
    'left_elbow',     # 7
    'right_elbow',    # 8
    'left_wrist',     # 9
    'right_wrist',    # 10
    'left_hip',       # 11
    'right_hip',      # 12
    'left_knee',      # 13
    'right_knee',     # 14
    'left_ankle',     # 15
    'right_ankle'     # 16
]

# Skeleton connections (edges in the graph)
COCO_SKELETON = [
    [0, 1], [0, 2],  # nose -> eyes
    [1, 3], [2, 4],  # eyes -> ears
    [0, 5], [0, 6],  # nose -> shoulders
    [5, 6],          # shoulders
    [5, 7], [7, 9],  # left arm
    [6, 8], [8, 10], # right arm
    [5, 11], [6, 12], # torso
    [11, 12],        # hips
    [11, 13], [13, 15], # left leg
    [12, 14], [14, 16]  # right leg
]

# Build adjacency matrix for GCN
def get_adjacency_matrix(num_keypoints=17):
    """
    Create adjacency matrix from skeleton definition.
    Returns normalized adjacency matrix (A_hat = D^-0.5 * A * D^-0.5)
    """
    adj = np.zeros((num_keypoints, num_keypoints))
    
    # Add edges
    for edge in COCO_SKELETON:
        adj[edge[0], edge[1]] = 1
        adj[edge[1], edge[0]] = 1
    
    # Add self-loops
    adj += np.eye(num_keypoints)
    
    # Normalize adjacency matrix (D^-0.5 * A * D^-0.5)
    degree = np.sum(adj, axis=1)
    degree_inv_sqrt = np.power(degree, -0.5)
    degree_inv_sqrt[np.isinf(degree_inv_sqrt)] = 0.
    D_inv_sqrt = np.diag(degree_inv_sqrt)
    adj_normalized = D_inv_sqrt @ adj @ D_inv_sqrt
    
    return torch.FloatTensor(adj_normalized)

# Get adjacency matrix
ADJ_MATRIX = get_adjacency_matrix(CONFIG['NUM_KEYPOINTS'])

print(f"Adjacency matrix shape: {ADJ_MATRIX.shape}")
print(f"Number of edges: {len(COCO_SKELETON)}")

## 4. Download Videos from S3

In [ ]:
def download_videos_from_s3(s3_prefix, local_dir, max_videos=None):
    """
    Download videos from S3 to local directory.
    """
    os.makedirs(local_dir, exist_ok=True)
    
    # List objects in S3
    paginator = s3_client.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=CONFIG['S3_BUCKET'], Prefix=s3_prefix)
    
    video_keys = []
    for page in pages:
        if 'Contents' in page:
            for obj in page['Contents']:
                key = obj['Key']
                if key.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
                    video_keys.append(key)
    
    if max_videos:
        video_keys = video_keys[:max_videos]
    
    print(f"Found {len(video_keys)} videos in {s3_prefix}")
    
    downloaded = []
    for key in tqdm(video_keys, desc=f"Downloading from {s3_prefix}"):
        filename = os.path.basename(key)
        local_path = os.path.join(local_dir, filename)
        
        # Skip if already downloaded
        if os.path.exists(local_path):
            downloaded.append(local_path)
            continue
        
        try:
            s3_client.download_file(CONFIG['S3_BUCKET'], key, local_path)
            downloaded.append(local_path)
        except Exception as e:
            print(f"Error downloading {key}: {e}")
    
    return downloaded

In [ ]:
# Download videos
print("Downloading violence videos...")
violence_videos = download_videos_from_s3(
    CONFIG['S3_VIOLENCE_PREFIX'],
    CONFIG['LOCAL_VIOLENCE_DIR']
)

print("\nDownloading normal videos...")
normal_videos = download_videos_from_s3(
    CONFIG['S3_NORMAL_PREFIX'],
    CONFIG['LOCAL_NORMAL_DIR']
)

# Prepare dataset
all_videos = violence_videos + normal_videos
all_labels = [1] * len(violence_videos) + [0] * len(normal_videos)

print(f"\nTotal dataset:")
print(f"  Violence videos: {len(violence_videos)}")
print(f"  Normal videos: {len(normal_videos)}")
print(f"  Total: {len(all_videos)}")

## 5. Load YOLO Pose Model

In [ ]:
# Load YOLOv8 pose model
print("Loading YOLOv8 Pose model...")
pose_model = YOLO('yolov8n-pose.pt')  # Nano model for speed
pose_model.to(CONFIG['DEVICE'])

print(f"Model loaded on {CONFIG['DEVICE']}")
print(f"Model type: YOLOv8 Pose (nano)")

## 6. Extract Pose Sequences from Videos

In [ ]:
def extract_pose_sequence(video_path, pose_model, config):
    """
    Extract pose keypoints from video.
    Returns: np.array of shape (T, M, K, 3) where:
        T = number of frames (SEQUENCE_LENGTH)
        M = max persons (MAX_PERSONS)
        K = keypoints (NUM_KEYPOINTS)
        3 = (x, y, confidence)
    """
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if fps == 0:
        fps = 30  # Default
    
    # Calculate frame interval for sampling
    frame_interval = max(1, int(fps / config['FPS_SAMPLE']))
    
    poses = []
    frame_idx = 0
    
    while len(poses) < config['SEQUENCE_LENGTH']:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Sample frames
        if frame_idx % frame_interval == 0:
            # Run pose detection
            results = pose_model(frame, verbose=False)
            
            # Extract keypoints
            frame_poses = np.zeros((config['MAX_PERSONS'], config['NUM_KEYPOINTS'], 3))
            
            if results[0].keypoints is not None and len(results[0].keypoints) > 0:
                keypoints = results[0].keypoints.data.cpu().numpy()  # (N, 17, 3)
                
                # Take up to MAX_PERSONS
                num_persons = min(len(keypoints), config['MAX_PERSONS'])
                
                for person_idx in range(num_persons):
                    kpts = keypoints[person_idx]  # (17, 3)
                    
                    # Normalize coordinates to [0, 1]
                    h, w = frame.shape[:2]
                    kpts[:, 0] /= w  # x
                    kpts[:, 1] /= h  # y
                    # confidence already in [0, 1]
                    
                    frame_poses[person_idx] = kpts
            
            poses.append(frame_poses)
        
        frame_idx += 1
    
    cap.release()
    
    # Pad or truncate to exact sequence length
    if len(poses) < config['SEQUENCE_LENGTH']:
        # Pad with zeros
        padding = config['SEQUENCE_LENGTH'] - len(poses)
        for _ in range(padding):
            poses.append(np.zeros((config['MAX_PERSONS'], config['NUM_KEYPOINTS'], 3)))
    elif len(poses) > config['SEQUENCE_LENGTH']:
        # Truncate
        poses = poses[:config['SEQUENCE_LENGTH']]
    
    return np.array(poses, dtype=np.float32)  # (T, M, K, 3)


def extract_and_cache_poses(video_paths, labels, pose_model, config, cache_path):
    """
    Extract poses from all videos and cache to disk.
    """
    if os.path.exists(cache_path):
        print(f"Loading cached poses from {cache_path}")
        data = np.load(cache_path, allow_pickle=True)
        return data['poses'], data['labels'], data['video_paths']
    
    print("Extracting poses from videos...")
    all_poses = []
    valid_labels = []
    valid_paths = []
    
    for video_path, label in tqdm(zip(video_paths, labels), total=len(video_paths)):
        try:
            pose_seq = extract_pose_sequence(video_path, pose_model, config)
            all_poses.append(pose_seq)
            valid_labels.append(label)
            valid_paths.append(video_path)
        except Exception as e:
            print(f"Error processing {video_path}: {e}")
            continue
    
    all_poses = np.array(all_poses)
    valid_labels = np.array(valid_labels)
    
    # Cache to disk
    print(f"Caching {len(all_poses)} pose sequences to {cache_path}")
    np.savez_compressed(
        cache_path,
        poses=all_poses,
        labels=valid_labels,
        video_paths=valid_paths
    )
    
    return all_poses, valid_labels, valid_paths

In [ ]:
# Extract poses (with caching)
cache_file = os.path.join(CONFIG['CACHE_DIR'], 'pose_sequences.npz')

poses, labels, video_paths = extract_and_cache_poses(
    all_videos,
    all_labels,
    pose_model,
    CONFIG,
    cache_file
)

print(f"\nExtracted pose sequences:")
print(f"  Shape: {poses.shape}")  # (N, T, M, K, 3)
print(f"  Memory: {poses.nbytes / 1024 / 1024:.2f} MB")
print(f"  Labels: {len(labels)}")

# Check for valid poses (at least some keypoints detected)
valid_mask = np.sum(poses[..., 2], axis=(1, 2, 3)) > 0  # Check confidence
print(f"  Videos with detected poses: {np.sum(valid_mask)} / {len(poses)}")

## 7. Train/Validation Split

In [ ]:
# Split data
X_train, X_val, y_train, y_val = train_test_split(
    poses,
    labels,
    train_size=CONFIG['TRAIN_VAL_SPLIT'],
    random_state=CONFIG['RANDOM_SEED'],
    stratify=labels
)

print(f"Training set: {len(X_train)} videos")
print(f"  Violence: {np.sum(y_train)} | Normal: {len(y_train) - np.sum(y_train)}")
print(f"\nValidation set: {len(X_val)} videos")
print(f"  Violence: {np.sum(y_val)} | Normal: {len(y_val) - np.sum(y_val)}")

## 8. PyTorch Dataset

In [ ]:
class PoseDataset(Dataset):
    """
    Dataset for pose sequences.
    Input: (T, M, K, 3) - T frames, M persons, K keypoints, 3 coords
    Output: (T, M*K, 3) - Flattened for GCN processing
    """
    def __init__(self, poses, labels, augment=False):
        self.poses = poses
        self.labels = labels
        self.augment = augment
    
    def __len__(self):
        return len(self.poses)
    
    def __getitem__(self, idx):
        pose = self.poses[idx]  # (T, M, K, 3)
        label = self.labels[idx]
        
        # Data augmentation
        if self.augment:
            pose = self._augment(pose)
        
        # Reshape: (T, M, K, 3) -> (T, M*K, 3)
        T, M, K, C = pose.shape
        pose = pose.reshape(T, M*K, C)
        
        return (
            torch.FloatTensor(pose),
            torch.LongTensor([label])[0]
        )
    
    def _augment(self, pose):
        """
        Simple augmentation: horizontal flip, temporal reverse
        """
        # Random horizontal flip
        if random.random() > 0.5:
            pose = pose.copy()
            pose[..., 0] = 1.0 - pose[..., 0]  # Flip x coordinates
            # Swap left-right keypoints
            # Left: [1,3,5,7,9,11,13,15] Right: [2,4,6,8,10,12,14,16]
            left_indices = [1, 3, 5, 7, 9, 11, 13, 15]
            right_indices = [2, 4, 6, 8, 10, 12, 14, 16]
            for m in range(pose.shape[1]):  # For each person
                temp = pose[:, m, left_indices].copy()
                pose[:, m, left_indices] = pose[:, m, right_indices]
                pose[:, m, right_indices] = temp
        
        # Random temporal reverse
        if random.random() > 0.5:
            pose = pose[::-1].copy()
        
        return pose


# Create datasets
train_dataset = PoseDataset(X_train, y_train, augment=True)
val_dataset = PoseDataset(X_val, y_val, augment=False)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['BATCH_SIZE'],
    shuffle=True,
    num_workers=CONFIG['NUM_WORKERS'],
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['BATCH_SIZE'],
    shuffle=False,
    num_workers=CONFIG['NUM_WORKERS'],
    pin_memory=True
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

# Test data loading
sample_pose, sample_label = next(iter(train_loader))
print(f"\nSample batch:")
print(f"  Pose shape: {sample_pose.shape}")  # (B, T, M*K, 3)
print(f"  Label shape: {sample_label.shape}")  # (B,)

## 9. Model Definition: ST-GCN + LSTM

In [ ]:
class GraphConvolution(nn.Module):
    """
    Simple Graph Convolution Layer.
    """
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        self.bias = nn.Parameter(torch.FloatTensor(out_features))
        self.reset_parameters()
    
    def reset_parameters(self):
        nn.init.xavier_uniform_(self.weight)
        nn.init.zeros_(self.bias)
    
    def forward(self, x, adj):
        """
        x: (B, N, in_features)
        adj: (N, N) adjacency matrix
        """
        # X * W
        support = torch.matmul(x, self.weight)  # (B, N, out_features)
        # A * X * W
        output = torch.matmul(adj, support)  # (B, N, out_features)
        output = output + self.bias
        return output


class SpatialTemporalGCN(nn.Module):
    """
    Spatial-Temporal Graph Convolutional Network.
    Processes pose sequences with graph structure.
    """
    def __init__(self, in_channels, hidden_dim, num_keypoints, max_persons):
        super().__init__()
        self.num_keypoints = num_keypoints
        self.max_persons = max_persons
        
        # Build separate graphs for each person
        self.gcn1 = GraphConvolution(in_channels, hidden_dim)
        self.gcn2 = GraphConvolution(hidden_dim, hidden_dim)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x, adj):
        """
        x: (B, T, M*K, C) where M=persons, K=keypoints, C=channels
        adj: (K, K) adjacency matrix for one person
        Returns: (B, T, M*K, hidden_dim)
        """
        B, T, N, C = x.shape
        
        # Process each person separately
        outputs = []
        for m in range(self.max_persons):
            # Extract one person's keypoints: (B, T, K, C)
            person_x = x[:, :, m*self.num_keypoints:(m+1)*self.num_keypoints, :]
            
            # Reshape for processing: (B*T, K, C)
            person_x = person_x.reshape(B*T, self.num_keypoints, C)
            
            # Apply GCN
            h = self.gcn1(person_x, adj)  # (B*T, K, hidden_dim)
            h = self.relu(h)
            h = self.dropout(h)
            
            h = self.gcn2(h, adj)  # (B*T, K, hidden_dim)
            h = self.relu(h)
            
            # Reshape back: (B, T, K, hidden_dim)
            h = h.reshape(B, T, self.num_keypoints, -1)
            outputs.append(h)
        
        # Concatenate all persons: (B, T, M*K, hidden_dim)
        output = torch.cat(outputs, dim=2)
        return output


class PoseViolenceDetector(nn.Module):
    """
    Full model: ST-GCN + LSTM + Classifier
    """
    def __init__(self, config, adj_matrix):
        super().__init__()
        self.config = config
        
        # Register adjacency matrix as buffer (not a parameter)
        self.register_buffer('adj_matrix', adj_matrix)
        
        # Spatial-Temporal GCN
        self.st_gcn = SpatialTemporalGCN(
            in_channels=3,  # (x, y, confidence)
            hidden_dim=config['GCN_HIDDEN_DIM'],
            num_keypoints=config['NUM_KEYPOINTS'],
            max_persons=config['MAX_PERSONS']
        )
        
        # Temporal modeling with LSTM
        # Input: flattened GCN features per frame
        lstm_input_dim = config['GCN_HIDDEN_DIM'] * config['NUM_KEYPOINTS'] * config['MAX_PERSONS']
        
        self.lstm = nn.LSTM(
            input_size=lstm_input_dim,
            hidden_size=config['LSTM_HIDDEN_DIM'],
            num_layers=config['LSTM_LAYERS'],
            batch_first=True,
            dropout=config['DROPOUT'] if config['LSTM_LAYERS'] > 1 else 0
        )
        
        # Classifier
        self.fc = nn.Sequential(
            nn.Linear(config['LSTM_HIDDEN_DIM'], config['LSTM_HIDDEN_DIM'] // 2),
            nn.ReLU(),
            nn.Dropout(config['DROPOUT']),
            nn.Linear(config['LSTM_HIDDEN_DIM'] // 2, config['NUM_CLASSES'])
        )
    
    def forward(self, x):
        """
        x: (B, T, M*K, 3)
        Returns: (B, num_classes)
        """
        B, T, N, C = x.shape
        
        # Spatial-Temporal GCN
        h = self.st_gcn(x, self.adj_matrix)  # (B, T, M*K, hidden_dim)
        
        # Flatten spatial features for LSTM
        h = h.reshape(B, T, -1)  # (B, T, M*K*hidden_dim)
        
        # LSTM temporal modeling
        lstm_out, _ = self.lstm(h)  # (B, T, lstm_hidden)
        
        # Use last timestep
        final_h = lstm_out[:, -1, :]  # (B, lstm_hidden)
        
        # Classification
        logits = self.fc(final_h)  # (B, num_classes)
        
        return logits

## 10. Initialize Model

In [ ]:
# Create model
model = PoseViolenceDetector(CONFIG, ADJ_MATRIX).to(CONFIG['DEVICE'])

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model: PoseViolenceDetector")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel architecture:")
print(model)

## 11. Training Setup

In [ ]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=CONFIG['LEARNING_RATE'],
    weight_decay=CONFIG['WEIGHT_DECAY']
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=5,
    verbose=True
)

# TensorBoard
writer = SummaryWriter(CONFIG['LOGS_DIR'])

# Training state
best_val_acc = 0.0
best_epoch = 0
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'val_f1': []
}

print("Training setup complete.")
print(f"Optimizer: Adam (lr={CONFIG['LEARNING_RATE']})")
print(f"Scheduler: ReduceLROnPlateau")
print(f"Loss: CrossEntropyLoss")

## 12. Training Loop

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc='Training')
    for poses, labels in pbar:
        poses = poses.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        # Forward
        logits = model(poses)
        loss = criterion(logits, labels)
        
        # Backward
        loss.backward()
        optimizer.step()
        
        # Metrics
        running_loss += loss.item()
        _, preds = torch.max(logits, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        pbar.set_postfix({
            'loss': running_loss / (pbar.n + 1),
            'acc': correct / total
        })
    
    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / total
    
    return epoch_loss, epoch_acc


def validate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        pbar = tqdm(loader, desc='Validation')
        for poses, labels in pbar:
            poses = poses.to(device)
            labels = labels.to(device)
            
            # Forward
            logits = model(poses)
            loss = criterion(logits, labels)
            
            # Metrics
            running_loss += loss.item()
            _, preds = torch.max(logits, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(loader)
    epoch_acc = accuracy_score(all_labels, all_preds)
    
    # Calculate F1
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='binary'
    )
    
    return epoch_loss, epoch_acc, f1, all_labels, all_preds

In [ ]:
# Training loop
print("\n" + "="*60)
print("Starting Training")
print("="*60 + "\n")

for epoch in range(CONFIG['NUM_EPOCHS']):
    print(f"\nEpoch {epoch+1}/{CONFIG['NUM_EPOCHS']}")
    print("-" * 60)
    
    # Train
    train_loss, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, CONFIG['DEVICE']
    )
    
    # Validate
    val_loss, val_acc, val_f1, val_labels, val_preds = validate_epoch(
        model, val_loader, criterion, CONFIG['DEVICE']
    )
    
    # Learning rate scheduling
    scheduler.step(val_acc)
    
    # Log metrics
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    
    writer.add_scalar('Loss/train', train_loss, epoch)
    writer.add_scalar('Loss/val', val_loss, epoch)
    writer.add_scalar('Accuracy/train', train_acc, epoch)
    writer.add_scalar('Accuracy/val', val_acc, epoch)
    writer.add_scalar('F1/val', val_f1, epoch)
    writer.add_scalar('LR', optimizer.param_groups[0]['lr'], epoch)
    
    # Print summary
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_f1': val_f1,
            'config': CONFIG
        }
        
        best_model_path = os.path.join(CONFIG['MODEL_SAVE_DIR'], 'best_pose_model.pth')
        torch.save(checkpoint, best_model_path)
        print(f"✓ New best model saved! (Acc: {val_acc:.4f})")
    
    # Save checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        checkpoint_path = os.path.join(
            CONFIG['MODEL_SAVE_DIR'], 
            f'checkpoint_epoch_{epoch+1}.pth'
        )
        torch.save(checkpoint, checkpoint_path)
        print(f"Checkpoint saved: {checkpoint_path}")

writer.close()

print("\n" + "="*60)
print("Training Complete!")
print(f"Best Val Accuracy: {best_val_acc:.4f} at Epoch {best_epoch}")
print("="*60)

## 13. Evaluation & Visualization

In [ ]:
# Load best model
checkpoint = torch.load(best_model_path)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Loaded best model from epoch {checkpoint['epoch']}")
print(f"Validation Accuracy: {checkpoint['val_acc']:.4f}")
print(f"Validation F1: {checkpoint['val_f1']:.4f}")

In [ ]:
# Final validation
val_loss, val_acc, val_f1, val_labels, val_preds = validate_epoch(
    model, val_loader, criterion, CONFIG['DEVICE']
)

print("\n" + "="*60)
print("Final Validation Results")
print("="*60)
print(f"Accuracy: {val_acc:.4f}")
print(f"F1 Score: {val_f1:.4f}")
print("\nClassification Report:")
print(classification_report(
    val_labels, val_preds,
    target_names=['Normal', 'Violence']
))

# Confusion matrix
cm = confusion_matrix(val_labels, val_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Violence'],
            yticklabels=['Normal', 'Violence'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['MODEL_SAVE_DIR'], 'confusion_matrix.png'))
plt.show()

print(f"\nConfusion matrix saved.")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(history['train_acc'], label='Train Acc')
axes[1].plot(history['val_acc'], label='Val Acc')
axes[1].plot(history['val_f1'], label='Val F1')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Score')
axes[1].set_title('Training and Validation Metrics')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['MODEL_SAVE_DIR'], 'training_history.png'))
plt.show()

print("Training history plots saved.")

## 14. Export Model for Deployment

In [ ]:
# Export to TorchScript
model.eval()
sample_input = torch.randn(
    1, 
    CONFIG['SEQUENCE_LENGTH'], 
    CONFIG['MAX_PERSONS'] * CONFIG['NUM_KEYPOINTS'], 
    3
).to(CONFIG['DEVICE'])

traced_model = torch.jit.trace(model, sample_input)
torchscript_path = os.path.join(CONFIG['MODEL_SAVE_DIR'], 'pose_model_torchscript.pt')
traced_model.save(torchscript_path)

print(f"TorchScript model saved: {torchscript_path}")

# Save configuration
config_path = os.path.join(CONFIG['MODEL_SAVE_DIR'], 'config.json')
with open(config_path, 'w') as f:
    # Convert non-serializable items
    save_config = CONFIG.copy()
    save_config['DEVICE'] = str(save_config['DEVICE'])
    json.dump(save_config, f, indent=2)

print(f"Configuration saved: {config_path}")

## 15. Upload to S3

In [ ]:
def upload_to_s3(local_path, s3_path):
    """Upload file to S3"""
    s3_parts = s3_path.replace('s3://', '').split('/', 1)
    bucket = s3_parts[0]
    key = s3_parts[1] if len(s3_parts) > 1 else ''
    
    s3_client.upload_file(local_path, bucket, key)
    print(f"Uploaded: {local_path} -> {s3_path}")

# Upload model files
print("Uploading model artifacts to S3...")

s3_model_path = f"s3://{CONFIG['S3_BUCKET']}/{prefix}/models/best_pose_model.pth"
upload_to_s3(best_model_path, s3_model_path)

upload_to_s3(
    torchscript_path,
    f"s3://{CONFIG['S3_BUCKET']}/{prefix}/models/pose_model_torchscript.pt"
)

upload_to_s3(
    config_path,
    f"s3://{CONFIG['S3_BUCKET']}/{prefix}/models/pose_config.json"
)

print("\n" + "="*60)
print("Model artifacts uploaded to S3!")
print(f"Model path: {s3_model_path}")
print("="*60)

## 16. Summary & Next Steps

In [ ]:
print("\n" + "="*80)
print(" "*20 + "TRAINING COMPLETE - SUMMARY")
print("="*80 + "\n")

print(f"📊 Dataset Statistics:")
print(f"   Total Videos: {len(all_videos)}")
print(f"   Training: {len(X_train)} | Validation: {len(X_val)}")
print(f"   Violence Videos: {np.sum(labels)} | Normal Videos: {len(labels) - np.sum(labels)}\n")

print(f"🎯 Best Model Performance:")
print(f"   Validation Accuracy: {checkpoint['val_acc']*100:.2f}%")
print(f"   Validation F1 Score: {checkpoint['val_f1']:.4f}")
print(f"   Achieved at Epoch: {checkpoint['epoch']}\n")

print(f"💾 Model Artifacts:")
print(f"   Best Model: {best_model_path}")
print(f"   TorchScript: {torchscript_path}")
print(f"   S3 Location: {s3_model_path}\n")

print(f"📈 Training Logs:")
print(f"   TensorBoard: {CONFIG['LOGS_DIR']}")
print(f"   View with: tensorboard --logdir={CONFIG['LOGS_DIR']}\n")

print("🚀 Advantages of Pose-Based Approach:")
print("   ✓ Lightweight: ~10x smaller than RGB+Flow models")
print("   ✓ Privacy-preserving: No visual appearance data")
print("   ✓ Efficient: Faster inference for real-time deployment")
print("   ✓ Robust: Better handling of occlusions and lighting\n")

print("📝 Next Steps:")
print("   1. Test model on additional validation videos")
print("   2. Integrate with real-time RTSP stream pipeline")
print("   3. Deploy to edge device (Jetson) or cloud")
print("   4. Set up Telegram alerting system")
print("   5. Monitor false positives and collect feedback")
print("   6. Retrain with new data periodically\n")

print("="*80 + "\n")

print("✅ All done! Your pose-based violence detection model is ready for deployment.")